In [ ]:
import pandas as pd
import openpyxl
import json
import os

def generate_excel_from_metadata(json_file, excel_file):
    """
    Generates an Excel file from the metadata described in a JSON file with adjusted column width, height, and style.
    
    :param json_file: Path to the input JSON file containing metadata.
    :param excel_file: Path to the output Excel file.
    """
    with open(json_file, 'r') as file:
        data = json.load(file)

    rows = []
    for tool in data:
        if tool['type'] == 'function':
            for param_name, param_info in tool['function']['parameters']['properties'].items():
                row = {
                    'Parameter Name': param_name,
                    'Type': param_info.get('type', ''),
                    'Description': param_info.get('description', ''),
                    'Examples': param_info.get('examples', ''),
                    'MaxLength': param_info.get('maxLength', ''),
                    'Enum': ', '.join(param_info['enum']) if 'enum' in param_info else ''
                }
                rows.append(row)
    
    df = pd.DataFrame(rows)
    
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df.to_excel(writer, index=False)
        workbook = writer.book
        worksheet = writer.sheets['Sheet1']
        
        for column_cells in worksheet.columns:
            length = max(len(str(cell.value)) for cell in column_cells)
            worksheet.column_dimensions[column_cells[0].column_letter].width = length
        for row in worksheet.iter_rows():
            row[0].style = 'Headline 1'
        
        worksheet.row_dimensions[1].height = 20
        
        for cell in worksheet["1:1"]:
            cell.style = 'Headline 1'
        
def generate_json_from_excel_metadata(excel_file, json_file, sheet_name='Sheet1', func_name='A_new_function', func_description='Extract information from the following clinical note'):
    df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl')
    
    data = df.to_dict(orient='records')
    properties = {}
    required = []
    for row in data:
        param_details = {'type': row.get('Type', '')}
        
        description = row.get('Description', '')
        examples = row.get('Examples', '')
        if not pd.isna(examples) and examples != '':
            description += f". Examples: {examples}"
        param_details['description'] = description
        
        maxLength = row.get('MaxLength', '')
        if not pd.isna(maxLength) and maxLength != '':
            param_details['maxLength'] = maxLength
        
        enum = row.get('Enum', '')
        if not pd.isna(enum) and enum != '':
            param_details['enum'] = [e.strip() for e in str(enum).split(',') if e.strip() != '']

        if param_details['type'] == 'array':
            item_type = row.get('ItemType', 'string')  
            item_description = row.get('ItemDescription', '')
            item_examples = row.get('ItemExamples', '')
            if not pd.isna(item_examples) and item_examples != '':
                item_description += f". Examples: {item_examples}"
                
            item_enum = row.get('ItemEnum', '')
            item_enum_details = []  
            if not pd.isna(item_enum) and item_enum != '':
                item_enum_details = [e.strip() for e in str(item_enum).split(',') if e.strip() != '']

            param_details['items'] = {
                'type': item_type,
                'description': item_description,
                'enum': item_enum_details
            }
            
            
        properties[row['Parameter Name']] = param_details
        required.append(row['Parameter Name'])

    structured_data = {
        'type': 'function',
        'function': {
            'name': func_name,
            'description': func_description,
            'parameters': {
                'type': 'object',
                'properties': properties,
                'required': required
            }
        }
    }
    
    with open(json_file, 'w') as file:
        json.dump(structured_data, file, indent=4)

input = "./GEARBOx Ontology.xlsx"
output = 'GEARBOx.json'
sheet = "Sheet1"

if output:
    generate_json_from_excel_metadata(input, output, sheet_name=sheet)
else:
    generate_json_from_excel_metadata(input, 
                                      os.path.splitext(os.path.basename(input))[0] + '_converted.json', 
                                      sheet_name=sheet)
print('finish')
